In [1]:
from ragwire import (
    MarkItDownLoader,
    get_markdown_splitter,
    get_splitter,
    get_embedding,
    QdrantStore,
    MetadataExtractor,
)

In [2]:
# Load a document
loader = MarkItDownLoader()
result = loader.load("../data/Apple_10k_2025.pdf")
print(f"Loaded: {result['file_name']}, chars: {len(result['text_content'])}")

Loaded: Apple_10k_2025.pdf, chars: 282696


In [3]:
result

{'text_content': 'UNITED STATESSECURITIES AND EXCHANGE COMMISSIONWashington, D.C. 20549FORM 10-K(Mark One)☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934For the fiscal year ended September 27, 2025or☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934For the transition period from              to             .Commission File Number: 001-36743Apple Inc.(Exact name of Registrant as specified in its charter)California94-2404110(State or other jurisdictionof incorporation or organization)(I.R.S. Employer Identification No.)One Apple Park WayCupertino, California95014(Address of principal executive offices)(Zip Code)(408) 996-1010(Registrant’s telephone number, including area code)Securities registered pursuant to Section 12(b) of the Act:Title of each classTrading symbol(s)Name of each exchange on which registeredCommon Stock, $0.00001 par value per shareAAPLThe Nasdaq Stock Market LLC0.000% Notes due 2025—The Nas

In [4]:
# Split into chunks
splitter = get_markdown_splitter(chunk_size=10000, chunk_overlap=2000)
chunks = splitter.split_text(result["text_content"])
print(f"Chunks: {len(chunks)}")

Chunks: 35


In [8]:
chunks[0][-2000:], chunks[1][:100]

('ntends,” “plans,” “predicts,” “will,” “would,” “could,” “can,” “may,” and similar terms. Forward-looking statements\nare not guarantees of future performance and the Company’s actual results may differ significantly from the results discussed in the forward-looking statements.\nFactors that might cause such differences include, but are not limited to, those discussed in Part I, Item 1A of this Form 10-K under the heading “Risk Factors.”\nThe Company assumes no obligation to revise or update any forward-looking statements for any reason, except as required by law.\n\nUnless  otherwise  stated,  all  information  presented  herein  is  based  on  the  Company’s  fiscal  calendar,  and  references  to  particular  years,  quarters,  months  or\nperiods refer to the Company’s fiscal years ended in September and the associated quarters, months and periods of those fiscal years. Each of the terms the\n“Company” and “Apple” as used herein refers collectively to Apple Inc. and its wholly own

In [12]:
# Create embeddings
embedding = get_embedding({
    "provider": "ollama",
    "model": "qwen3-embedding:0.6b",
    "base_url": "http://192.168.1.9:11434",
})
vector = embedding.embed_query("test query")
print(f"Embedding dimension: {len(vector)}")


Embedding dimension: 1024


In [13]:
# Connect to vector store
store = QdrantStore(
    config={"url": "http://192.168.1.9:6333"},
    embedding=embedding,
    collection_name="financial_docs",
)
vectorstore = store.get_store(use_sparse=True)
results = vectorstore.similarity_search("total revenue", k=3)
print(f"Retrieved: {len(results)} chunks")

Retrieved: 3 chunks
